# 002 Source-Specific Preprocessing Gate

This notebook is the next forensic checkpoint after source inventory and safety audit. It does not train, index, or generate answers. Its job is to turn source-level gates into source-specific use lanes so we can maximize the current data without mixing unsafe, unreviewed, or commercial-unsafe records into downstream stages.

Primary outputs are written under `data/processed/source_specific_preprocessing_gate/`.

## Gate Contract

- Raw data stays immutable.
- Public ticket datasets remain blocked from training until PII, license, provenance, and schema review are complete.
- Stack Exchange records remain share-alike constrained and require record-level security filtering before retrieval or classifier use.
- Security and firmware records may become evaluation or escalation-safety fixtures even when blocked from answer generation.
- Use lanes are additive, not blunt allow/block decisions.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

DATA = ROOT / 'data'
OUT = DATA / 'processed' / 'source_specific_preprocessing_gate'
PYTHON = ROOT / '.it_support' / 'Scripts' / 'python.exe'

ROOT

## Run Offline Gate

The gate script is offline and deterministic over the current raw files. Set `RUN_GATE = True` to regenerate the artifacts after raw data or gate logic changes.

In [ ]:
RUN_GATE = False

cmd = [str(PYTHON if PYTHON.exists() else sys.executable), 'scripts/run_preprocessing_gate.py']
print(' '.join(cmd))

if RUN_GATE:
    completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
else:
    print('Gate not rerun. Reading existing artifacts.')

## Summary

In [ ]:
summary_path = OUT / 'gate_summary.md'
display(Markdown(summary_path.read_text(encoding='utf-8')))

summary = json.loads((OUT / 'gate_summary.json').read_text(encoding='utf-8'))
summary['policy']

## Stack Exchange Record Gate

This table contains one row per distinct question record from the scaled Stack Exchange run. It keeps ranked multi-label signals, safety flags, attribution-preserving source URLs, and use lanes.

In [ ]:
stack_gate = pd.read_csv(OUT / 'stack_exchange_scaled_run_gate.csv')
display(stack_gate.groupby('gate_decision').size().rename('records').reset_index().sort_values('gate_decision'))
display(stack_gate.groupby('primary_domain').size().rename('records').reset_index().sort_values('records', ascending=False))

stack_gate.head(10)

In [ ]:
review_cols = ['record_id', 'title', 'question_tags', 'query_tags', 'primary_domain', 'secondary_domains', 'gate_decision', 'use_lanes', 'source_url']
needs_review = stack_gate[stack_gate['gate_decision'] != 'candidate_after_license_review'][review_cols]
needs_review.head(25)

## Ticket Dataset Intake

Ticket datasets are inventoried for schema, declared license, declared PII/synthetic posture, possible PII/secrets in bounded samples, label columns, text columns, and provisional use lanes. They remain blocked from training until manual review promotes a subset.

In [ ]:
ticket_intake = pd.read_csv(OUT / 'ticket_dataset_intake.csv')
ticket_profiles = pd.read_csv(OUT / 'ticket_dataset_file_profiles.csv')

display(ticket_intake[['dataset_id', 'declared_license', 'declared_pii_removed', 'declared_synthetic', 'text_file_count', 'label_file_count', 'pii_signal_count', 'secret_like_signal_count', 'provisional_use_lanes']])
display(ticket_profiles[ticket_profiles['label_columns'].fillna('').astype(str) != ''][['dataset_id', 'path', 'text_columns', 'label_columns', 'row_count', 'provisional_use_lanes']].head(30))

## Redacted Previews

These previews are for human inspection only. They are redacted snippets, not normalized training records.

In [ ]:
stack_preview = pd.read_json(OUT / 'stack_exchange_gated_preview.jsonl', lines=True)
ticket_preview = pd.read_json(OUT / 'ticket_dataset_redacted_preview.jsonl', lines=True)

display(stack_preview[['record_id', 'title', 'gate_decision', 'use_lanes']].head(10))
display(ticket_preview[['dataset_id', 'path', 'preview', 'provisional_use_lanes']].head(10))

## Exit Criteria

Before moving to normalization, choose one source-specific lane and document the promotion rule. Recommended first promotions:

1. Stack Exchange `candidate_after_license_review` records into a normalized POC classifier/retrieval candidate set with attribution attached.
2. Stack Exchange security and firmware records into safety/evaluation fixtures, not answer-generation records.
3. Ticket datasets with clear license/provenance and acceptable redaction posture into classifier-only candidates after manual sample review.